# 🧠 Handwritten Digit Classification using CNN (PyTorch)

## 📌 Project Overview
This project implements a **Convolutional Neural Network (CNN)** using **PyTorch** to classify handwritten digits (0–9) from the **MNIST dataset**.  
The model learns spatial patterns such as edges, curves, and shapes from grayscale images.

---

## 📊 Dataset
- **Name:** MNIST
- **Images:** 28 × 28 grayscale
- **Classes:** 10 (digits 0–9)
- **Training samples:** 60,000
- **Test samples:** 10,000

---

## 🏗️ Model Architecture

### Feature Extraction (CNN)
- `Conv2d(1 → 8, kernel=3, padding=1)`
- `ReLU`
- `MaxPool2d(2)`
- `Conv2d(8 → 16, kernel=3, padding=1)`
- `ReLU`
- `MaxPool2d(2)`

**Output feature map size:** `16 × 7 × 7`

---

### Classification (Fully Connected)
- `Linear(16×7×7 → 64)`
- `ReLU`
- `Linear(64 → 10)`

---

## ⚙️ Training Details
- **Loss Function:** CrossEntropyLoss
- **Optimizer:** Adam
- **Learning Rate:** 0.001
- **Batch Size:** 64
- **Epochs:** 3
- **Device:** CPU / GPU (CUDA if available)

---

## 🔁 Training Process
1. Load MNIST dataset using `torchvision`
2. Apply `ToTensor()` transformation
3. Train CNN using mini-batch gradient descent
4. Backpropagate loss and update weights
5. Evaluate model on test dataset

---

## 📈 Evaluation
- Model performance is evaluated using **classification accuracy** on the test set.
- Typical accuracy after 3 epochs: **~97–98%**

---

## 💾 Model Saving
The trained model weights are saved as:


In [12]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

# -----------------------
# Device
# -----------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -----------------------
# Transform
# -----------------------
transform = transforms.ToTensor()

# -----------------------
# Dataset
# -----------------------
train_data = MNIST(
    root="data",
    train=True,
    download=True,
    transform=transform
)

test_data = MNIST(
    root="data",
    train=False,
    download=True,
    transform=transform
)

# -----------------------
# DataLoader
# -----------------------
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# -----------------------
# Model
# -----------------------
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # flatten
        x = self.classifier(x)
        return x

# -----------------------
# Initialize model
# -----------------------
model = SimpleCNN().to(device)

# -----------------------
# Loss & Optimizer
# -----------------------
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# -----------------------
# Training
# -----------------------
epochs = 3

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f}")

# -----------------------
# Testing
# -----------------------
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")

# -----------------------
# Save model
# -----------------------
torch.save(model.state_dict(), "simple_cnn_mnist.pth")
print("Model saved as 'simple_cnn_mnist.pth'")


Using device: cpu
Epoch [1/3] | Loss: 0.3360
Epoch [2/3] | Loss: 0.0930
Epoch [3/3] | Loss: 0.0645
Test Accuracy: 98.23%
Model saved as 'simple_cnn_mnist.pth'
